## Langchain memory - Conversation Context

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

model = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.5,
        "max_new_tokens": 100
    }
)

llm = ChatHuggingFace(llm = model)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful study assistant."),
    MessagesPlaceholder(variable_name="history"),   # past messages are injected here
    ("human", "{input}"),
])




In [ ]:
chain = prompt | llm | StrOutputParser()

# One history object per session id
_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]

chat = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

cfg = {"configurable": {"session_id": "sam-1"}}

print(chat.invoke({"input": "Hi, I'm Sam and I'm studying decision trees."}, config=cfg))
print(chat.invoke({"input": "What topic did I say I was studying?"}, config=cfg))
# -> "You said you're studying decision trees."


In [ ]:
# A different session has a separate, empty memory
print(chat.invoke({"input": "What topic am I studying?"},
                  config={"configurable": {"session_id": "other"}}))